# PC-LPUF and XOR-PUF Reliability Attack

This notebook performs reliability-based attacks on both PC-LPUF and XOR-PUF designs.
It generates the underlying APUF and REAP-NVM components, simulates noisy evaluations, and analyzes reliability behavior using metrics such as flip rate and correlation.

The workflow includes:
- a reliability attack for the PC-LPUF,
- a reliability attack for the XOR-PUF,
- reliability analysis for the full PCLPUF architectures,
- configurable settings such as the number of PUF components and dataset size so the reported paper results can be reproduced.


In [ ]:
##########PCLPUF Reliability Attack Script (CMA-ES)##########

import numpy as np
from cma import CMAEvolutionStrategy
import csv
import os

# ─── Parity Transform ─────────────────────────────────────────────────────────
def transform(challenges, chal_size):
    c   = 1 - 2 * challenges.astype(float)
    N   = challenges.shape[0]
    Phi = np.ones((N, chal_size + 1))
    for i in range(chal_size):
        Phi[:, i] = np.prod(c[:, i:], axis=1)
    return Phi

# ─── APUF Generation ──────────────────────────────────────────────────────────
def apuf_generate(k, chal_size, sigma=0.05):
    return np.random.normal(0, 0.05, (k, chal_size + 1))

# ─── APUF Response ────────────────────────────────────────────────────────────
def apuf_response(w, Phi):
    delta = Phi @ w
    return np.where(delta > 0, 0, 1).astype(np.int8)

# ─── APUF Noisy Response ──────────────────────────────────────────────────────
def apuf_response_noisy(w, Phi, sigma_noise_apuf, sigma=1):
    """Additive noise scaled by sigma — same model as XOR PUF."""
    noise = np.random.normal(0, sigma_noise_apuf * sigma, w.shape)
    return apuf_response(w + noise, Phi)

# ─── REAP-NVM Generation ─────────────────────────────────────────────────────
def ReapNVM(num_bits, seed, sigma_proc = 0.05):
    rng = np.random.default_rng(seed)

    chal_length = num_bits
    n_levels = 4

    # ---- Fixed nominal resistance levels ----
    R_levels_nom = np.array([10e3, 75e3, 125e3, 275e3])

    # ---- Convert to log domain ----
    log_R_levels_nom = np.log10(R_levels_nom)

    # ---- Allocate output arrays ----
    tR = np.zeros((2, n_levels, chal_length))

    # ---- Generate per-cell values ----
    for row in range(2):
        for stage in range(chal_length):

            # Add Gaussian process variation in log domain
            log_levels = log_R_levels_nom + sigma_proc * rng.standard_normal(n_levels)

            # Convert back to linear domain
            levels = 10 ** log_levels

            # RC delay mapping
            tR[row, :, stage] = levels * 250e-12

    # ---- Deterministic switching delay ----
    tSW = np.full((2, 2, chal_length), 372.0 / 1e12)

    return 4.0 * tR, 4.0 * tSW

# ─── REAP-NVM Evaluate ────────────────────────────────────────────────────────
def ReapNVM_evaluate(PUF, challenge, position, value, chunk_size=100_000):
    chal      = (challenge + 1) / 2.0
    tR4, tSW4 = PUF
    chalpos   = position.astype(int).flatten()
    chalval   = value.astype(int).flatten()
    num_challenges, _ = chal.shape
    responses = np.zeros(num_challenges, dtype=np.int8)
    for start in range(0, num_challenges, chunk_size):
        end        = min(start + chunk_size, num_challenges)
        N          = end - start
        chal_chunk = chal[start:end, :]
        pos_chunk  = chalpos[start:end]
        val_chunk  = chalval[start:end]
        tv1        = np.tile(tR4[0, 0, :], (N, 1))
        tv2        = np.tile(tR4[1, 0, :], (N, 1))
        tv1[np.arange(N), pos_chunk] = tR4[0, val_chunk, pos_chunk]
        tv2[np.arange(N), pos_chunk] = tR4[1, val_chunk, pos_chunk]
        c   = np.bitwise_xor.accumulate(chal_chunk.astype(np.uint8), axis=1)
        t1  = np.sum(np.where(c==0, tv1+tSW4[0,0,:], tv2+tSW4[1,0,:]), axis=1)
        t2  = np.sum(np.where(c==0, tv2+tSW4[0,1,:], tv1+tSW4[1,1,:]), axis=1)
        responses[start:end] = (t1 > t2).astype(np.int8)
    return responses

# ─── REAP-NVM Noisy Evaluate ─────────────────────────────────────────────────
def ReapNVM_evaluate_noisy(PUF, challenge, position, value, sigma_noise=0.2):
    """
    Evaluate REAP-NVM with additive log-domain Gaussian noise.
    - PUF: tuple (tR, tSW) from ReapNVM
    - sigma_noise: standard deviation in log10 domain (same as sigma_proc)
    """
    tR_nom, tSW = PUF
    rows, n_levels, n_cells = tR_nom.shape

    # Convert to log10 domain
    log_tR = np.log10(tR_nom)

    # Generate Gaussian noise in log-domain (same σ for all levels)
    log_noise = sigma_noise * np.random.standard_normal((rows, n_levels, n_cells))

    # Apply noise
    log_tR_noisy = log_tR + log_noise

    # Convert back to linear domain
    tR_noisy = 10 ** log_tR_noisy

    # Evaluate with noisy tR
    return ReapNVM_evaluate((tR_noisy, tSW), challenge, position, value)

# ─── Theta Obfuscation ────────────────────────────────────────────────────────
def dec_to_bin_vec(x, bitlen):
    return np.array([(x >> i) & 1 for i in range(bitlen)][::-1], dtype=np.uint8)

def bin_vec_to_dec(bits):
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out

def sliding_window_xor(bits, window_bits):
    P = bits.shape[0]
    U = window_bits.shape[0]
    window_bits = window_bits.astype(bits.dtype)
    for start in range(0, P, U):
        end = min(start + U, P)
        w = end - start
        bits[start:end] ^= window_bits[:w]
    return bits

def xor_obfuscate_position_value(position, value, upper_resp):
    N = position.shape[0]
    pos_out = np.zeros(N, dtype=np.uint32)
    val_out = np.zeros(N, dtype=np.uint32)
    for n in range(N):
        window_bits = upper_resp[:, n]
        pos_bits    = dec_to_bin_vec(position[n], 7)
        val_bits    = dec_to_bin_vec(value[n], 2)
        pos_bits    = sliding_window_xor(pos_bits, window_bits)
        val_bits    = sliding_window_xor(val_bits, window_bits)
        pos_out[n]  = bin_vec_to_dec(pos_bits)
        val_out[n]  = bin_vec_to_dec(val_bits)
    return pos_out, val_out

# ─── PCLPUF Generation ───────────────────────────────────────────────────────
def pclpuf_generate(chal_size, k_up, k_d, seed=0):
    np.random.seed(seed)
    upper_w    = apuf_generate(k_up, chal_size, sigma=1)
    lower_pufs = [ReapNVM(chal_size, seed=seed+k) for k in range(k_d)]
    return upper_w, lower_pufs

# ─── PCLPUF Clean Evaluate ───────────────────────────────────────────────────
def pclpuf_evaluate(upper_w, lower_pufs, challenges, position, value):
    k_up   = upper_w.shape[0]
    N      = challenges.shape[0]
    Phi_up = transform(challenges, challenges.shape[1])

    binary_arr = np.zeros((k_up, N), dtype=np.int8)
    for i in range(k_up):
        binary_arr[i] = apuf_response(upper_w[i], Phi_up)

    eff_pos, eff_val = xor_obfuscate_position_value(position, value, binary_arr)

    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        resp     = ReapNVM_evaluate(puf, challenges, eff_pos, eff_val)
        xor_resp = np.bitwise_xor(xor_resp, resp)

    return xor_resp

# ─── PCLPUF Noisy Evaluate ───────────────────────────────────────────────────
def pclpuf_evaluate_noisy(upper_w, lower_pufs, challenges, position, value,
                           sigma_noise_apuf, sigma_noise_reap):
    k_up   = upper_w.shape[0]
    N      = challenges.shape[0]
    Phi_up = transform(challenges, challenges.shape[1])

    binary_arr = np.zeros((k_up, N), dtype=np.int8)
    for i in range(k_up):
        binary_arr[i] = apuf_response_noisy(upper_w[i], Phi_up, sigma_noise_apuf)

    eff_pos, eff_val = xor_obfuscate_position_value(position, value, binary_arr)

    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        resp     = ReapNVM_evaluate_noisy(puf, challenges, eff_pos, eff_val, sigma_noise_reap)
        xor_resp = np.bitwise_xor(xor_resp, resp)

    return xor_resp

# ─── Compute Noisy Responses ──────────────────────────────────────────────────
def compute_noisy_responses_pclpuf(upper_w, lower_pufs, challenges,
                                    position, value, sigma_noise_apuf,
                                    sigma_noise_reap, evaluations):
    N         = challenges.shape[0]
    responses = np.zeros((N, evaluations), dtype=int)
    for e in range(evaluations):
        responses[:, e] = pclpuf_evaluate_noisy(
            upper_w, lower_pufs, challenges, position, value,
            sigma_noise_apuf, sigma_noise_reap
        )
    return responses

# ─── Compute Noise Information ────────────────────────────────────────────────
def compute_noise_information(responses, evaluations):
    row_sums = responses.sum(axis=1).astype(float)
    return np.abs(evaluations / 2.0 - row_sums)

# ─── Model Accuracy ───────────────────────────────────────────────────────────
def model_acc(w, Phi, responses):
    preds = np.where(Phi @ w > 0, 0, 1)
    return np.mean(preds == responses)

# ─── Fitness Function ─────────────────────────────────────────────────────────
def model_acc_rel(w, Phi, inform):
    delta = Phi @ w
    corr  = np.corrcoef(np.abs(delta), inform)[0, 1]
    return 0.0 if np.isnan(corr) else corr

# ─── CMA-ES ───────────────────────────────────────────────────────────────────
def cmaes_attack(Phi, inform, size, stop_eval=30000, seed=None):
    if seed is not None:
        np.random.seed(seed)
    x0 = np.random.rand(size)
    es = CMAEvolutionStrategy(x0, 0.5, {
        'maxfevals': stop_eval,
        'tolx':      1e-10,
        'verbose':   -9,
    })
    while not es.stop():
        solutions = es.ask()
        fitnesses = [-model_acc_rel(np.array(s), Phi, inform) for s in solutions]
        es.tell(solutions, fitnesses)
    return np.array(es.result.xbest)


# ─── Flip-Rate Analysis (integrated from diagnostic script) ─────────────────
def compute_flip_rate(upper_w, lower_pufs, challenges, position, value):
    """
    Measures how often the final PC-LPUF output flips when every upper-layer
    APUF response is inverted (upper_resp -> 1 - upper_resp), holding the
    challenge, position, and value fixed.

    ~50% flip rate  -> upper layer has no deterministic influence on output
    far from 50%    -> upper layer's decision leaks into the final response
                       (this is the property a reliability-based attack,
                       e.g. CMA-ES, can try to exploit)
    """
    k_up   = upper_w.shape[0]
    N      = challenges.shape[0]
    Phi_up = transform(challenges, challenges.shape[1])

    upper_resp = np.zeros((k_up, N), dtype=np.int8)
    for i in range(k_up):
        upper_resp[i] = apuf_response(upper_w[i], Phi_up)
    upper_resp_flipped = 1 - upper_resp

    eff_pos_clean,   eff_val_clean   = xor_obfuscate_position_value(position, value, upper_resp)
    eff_pos_flipped, eff_val_flipped = xor_obfuscate_position_value(position, value, upper_resp_flipped)

    resp_clean   = np.zeros(N, dtype=np.int8)
    resp_flipped = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        resp_clean   = np.bitwise_xor(resp_clean,   ReapNVM_evaluate(puf, challenges, eff_pos_clean,   eff_val_clean))
        resp_flipped = np.bitwise_xor(resp_flipped, ReapNVM_evaluate(puf, challenges, eff_pos_flipped, eff_val_flipped))

    flip_mask = (resp_clean != resp_flipped)
    return np.mean(flip_mask), flip_mask


# ─── Match-Rate Reliability (consistent with XOR-PUF script's definition) ───
def compute_apuf_reliability(w, Phi, sigma_noise_apuf, rel_reps=1000):
    """
    Rel = fraction of noisy re-evaluations that match the noiseless response,
    averaged over rel_reps independent noise draws. Same definition used for
    the standalone XOR-PUF reliability attack script, so numbers are comparable.
    """
    ref = apuf_response(w, Phi)
    matches = np.zeros(Phi.shape[0], dtype=float)
    for _ in range(rel_reps):
        noisy = apuf_response_noisy(w, Phi, sigma_noise_apuf)
        matches += (noisy == ref)
    return (matches / rel_reps).mean()


def compute_reap_reliability(puf, challenges, position, value, sigma_noise_reap, rel_reps=1000):
    ref = ReapNVM_evaluate(puf, challenges, position, value)
    matches = np.zeros(challenges.shape[0], dtype=float)
    for _ in range(rel_reps):
        noisy = ReapNVM_evaluate_noisy(puf, challenges, position, value, sigma_noise_reap)
        matches += (noisy == ref)
    return (matches / rel_reps).mean()


def compute_pclpuf_reliability(upper_w, lower_pufs, challenges, position, value,
                                sigma_noise_apuf, sigma_noise_reap, rel_reps=1000):
    ref = pclpuf_evaluate(upper_w, lower_pufs, challenges, position, value)
    matches = np.zeros(challenges.shape[0], dtype=float)
    for _ in range(rel_reps):
        noisy = pclpuf_evaluate_noisy(upper_w, lower_pufs, challenges, position, value,
                                       sigma_noise_apuf, sigma_noise_reap)
        matches += (noisy == ref)
    return (matches / rel_reps).mean()

# ─── Main ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Experiment configuration parameters
    # chal_size: number of challenge bits for each PUF instance
    # k_up: number of upper-layer APUF components
    # k_d: number of lower-layer REAP-NVM components
    # sigma_noise: noise level applied to both APUF and REAP-NVM responses
    # evaluations: number of noisy evaluations used to build the reliability signal
    # n_train: number of training challenges used for CMA-ES optimization
    # n_test: number of test challenges used for accuracy evaluation
    # max_runs: maximum number of CMA-ES attack runs to execute
    # threshold: accuracy threshold for declaring an APUF recovered
    # size_up: dimensionality of the upper-layer weight vector
    
    # we evaluated the following configurations in the paper:
    # chal_size = 128, k_up = 1, k_d = 1
    # chal_size = 128, k_up = 3, k_d = 1

    chal_size        = 128
    k_up             = 1 ## number of upper APUFs
    k_d              = 1 ### number of upper lower REAP-NVMs
    sigma_noise      = 0.2 # 0.05 for low noise, 0.1 for high noise
    sigma_noise_apuf = 0.05 * sigma_noise
    sigma_noise_reap = 0.05 * sigma_noise
    evaluations      = 11
    n_train          = 300_000
    n_test           = 500
    max_runs         = 50
    threshold        = 0.9
    size_up          = chal_size + 1

    np.random.seed(0)

    upper_w, lower_pufs = pclpuf_generate(chal_size, k_up, k_d, seed=42)

    TrS = np.random.randint(0, 2, (n_train, chal_size))
    TeS = np.random.randint(0, 2, (n_test,  chal_size))

    position_train = np.random.randint(0, chal_size, n_train)
    value_train    = np.random.randint(0, 4,         n_train)
    position_test  = np.random.randint(0, chal_size, n_test)
    value_test     = np.random.randint(0, 4,         n_test)

    Phi_TrS = transform(TrS, chal_size)
    Phi_TeS = transform(TeS, chal_size)

    indiv_resp_up = {i: apuf_response(upper_w[i], Phi_TeS) for i in range(k_up)}

    print("Computing clean responses...")
    clean_resp = pclpuf_evaluate(upper_w, lower_pufs, TeS, position_test, value_test)

    # ── Per-component reliability (match-rate, same definition as XOR-PUF script) ──
    # Use a dedicated reliability challenge set, separate from train/test, matching
    # the standalone XOR-PUF script's rel_size=500 / rel_reps=1000 convention so the
    # numbers are directly comparable across scripts.
    rel_size = 500
    rel_reps = 1000
    C_rel        = np.random.randint(0, 2, (rel_size, chal_size))
    position_rel = np.random.randint(0, chal_size, rel_size)
    value_rel    = np.random.randint(0, 4,         rel_size)
    Phi_rel      = transform(C_rel, chal_size)

    print(f"Computing per-component reliability ({rel_reps} noise draws, {rel_size} challenges)...")

    print("  Upper APUFs (individual):")
    apuf_reliabilities = []
    for i in range(k_up):
        rel_i = compute_apuf_reliability(upper_w[i], Phi_rel, sigma_noise_apuf, rel_reps)
        apuf_reliabilities.append(rel_i)
        print(f"    APUF {i+1}: Reliability = {rel_i*100:.2f}%  (sigma_noise_apuf={sigma_noise_apuf})")

    print("  Lower REAP-NVM (individual, using clean upper responses):")
    upper_resp_rel = np.zeros((k_up, rel_size), dtype=np.int8)
    for i in range(k_up):
        upper_resp_rel[i] = apuf_response(upper_w[i], Phi_rel)
    eff_pos_rel, eff_val_rel = xor_obfuscate_position_value(
        position_rel, value_rel, upper_resp_rel
    )
    for k, puf in enumerate(lower_pufs):
        rel_k = compute_reap_reliability(
            puf, C_rel, eff_pos_rel, eff_val_rel, sigma_noise_reap, rel_reps
        )
        print(f"    REAP-NVM {k+1}: Reliability = {rel_k*100:.2f}%  (sigma_noise_reap={sigma_noise_reap})")

    # ── Full PC-LPUF ──────────────────────────────────────────────────────────
    print("Computing full PC-LPUF reliability...")
    reliability = compute_pclpuf_reliability(
        upper_w, lower_pufs, C_rel, position_rel, value_rel,
        sigma_noise_apuf, sigma_noise_reap, rel_reps
    )
    print(f"PC-LPUF Overall Reliability: {reliability*100:.2f}%")

    # ── CMA-ES training signal (separate from the reliability numbers above) ──
    # This "inform" statistic is only a noisy proxy used to drive the CMA-ES
    # attack's fitness function; it is NOT the reliability metric itself.
    upper_resp_tr = np.zeros((k_up, n_train), dtype=np.int8)
    for i in range(k_up):
        upper_resp_tr[i] = apuf_response(upper_w[i], Phi_TrS)
    eff_pos_tr, eff_val_tr = xor_obfuscate_position_value(
        position_train, value_train, upper_resp_tr
    )

    noisy_resp = compute_noisy_responses_pclpuf(
        upper_w, lower_pufs, TrS, position_train, value_train,
        sigma_noise_apuf, sigma_noise_reap, evaluations
    )
    inform = compute_noise_information(noisy_resp, evaluations)

    # ── Correlation Analysis ──────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("Correlation Analysis")
    print("=" * 55)

    delta_up = Phi_TrS @ upper_w[0]

    tR4, tSW4  = lower_pufs[0]
    chal_c     = (TrS + 1) / 2.0
    chalpos    = eff_pos_tr.astype(int)
    chalval    = eff_val_tr.astype(int)
    delta_reap = np.zeros(n_train)
    chunk_size = 100_000
    for start in range(0, n_train, chunk_size):
        end    = min(start + chunk_size, n_train)
        N      = end - start
        cc     = chal_c[start:end]
        pc, vc = chalpos[start:end], chalval[start:end]
        tv1    = np.tile(tR4[0, 0, :], (N, 1))
        tv2    = np.tile(tR4[1, 0, :], (N, 1))
        tv1[np.arange(N), pc] = tR4[0, vc, pc]
        tv2[np.arange(N), pc] = tR4[1, vc, pc]
        c  = np.bitwise_xor.accumulate(cc.astype(np.uint8), axis=1)
        t1 = np.sum(np.where(c==0, tv1+tSW4[0,0,:], tv2+tSW4[1,0,:]), axis=1)
        t2 = np.sum(np.where(c==0, tv2+tSW4[0,1,:], tv1+tSW4[1,1,:]), axis=1)
        delta_reap[start:end] = t1 - t2

    corr_up   = np.corrcoef(np.abs(delta_up),   inform)[0, 1]
    corr_reap = np.corrcoef(np.abs(delta_reap), inform)[0, 1]
    print(f"  corr(|delta_up|,   inform): {corr_up:.6f}")
    print(f"  corr(|delta_reap|, inform): {corr_reap:.6f}")
    print(f"  (higher = that layer dominates the reliability signal)")

    # ── Flip-Rate Analysis ─────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("Flip-Rate Analysis (upper-layer influence on final output)")
    print("=" * 55)
    flip_rate, flip_mask = compute_flip_rate(
        upper_w, lower_pufs, TrS, position_train, value_train
    )
    print(f"  Overall flip rate: {flip_rate*100:.2f}%")
    print(f"  (~50% => upper layer has no deterministic influence on output)")

    # ── CMA-ES attack on upper layer ──────────────────────────────────────────
    print("\n" + "=" * 55)
    print(f"Attacking Upper Layer ({k_up} independent APUFs)")
    print("=" * 55)

    found        = [False] * k_up
    found_at_run = [None]  * k_up
    best_models  = [None]  * k_up

    # ── Prepare CSV file for results ──────────────────────────────────────────
    results_file = f'pclpuf_reliability_attack_KUP{k_up}_Kd{k_d}_train{n_train}.csv'
    file_exists  = os.path.isfile(results_file)

    # Header: fixed columns + one column per APUF accuracy
    apuf_acc_headers = [f'apuf_{i+1}_acc' for i in range(k_up)]
    header = (['chal_size', 'k_up', 'k_d', 'sigma_noise_apuf', 'sigma_noise_reap',
               'evaluations', 'n_train', 'n_test', 'threshold', 'reliability_pclpuf',
               'flip_rate', 'run', 'best_apuf'] + apuf_acc_headers + ['best_acc', 'found'])

    with open(results_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(header)

    for run in range(max_runs):
        w_model = cmaes_attack(Phi_TrS, inform, size_up, seed=run)

        accs = []
        for i in range(k_up):
            acc = model_acc(w_model, Phi_TeS, indiv_resp_up[i])
            acc = max(acc, 1 - acc)
            accs.append(acc)

        best_i   = int(np.argmax(accs))
        best_acc = accs[best_i]

        print(f"Run {run+1:3d} | Accs: {[f'{a:.3f}' for a in accs]} "
              f"| Best: APUF {best_i+1} ({best_acc:.3f})", end="")

        if best_acc >= threshold and not found[best_i]:
            found[best_i]        = True
            found_at_run[best_i] = run + 1
            best_models[best_i]  = w_model
            print(f"  ✓ UPPER APUF {best_i+1} FOUND", end="")
        print()

        # ── Write this run to CSV ──────────────────────────────────────────────
        with open(results_file, 'a', newline='') as f:
            writer = csv.writer(f)
            acc_values = [f'{a*100:.4f}' for a in accs]
            row = ([chal_size, k_up, k_d, sigma_noise_apuf, sigma_noise_reap,
                    evaluations, n_train, n_test, threshold, f'{reliability*100:.4f}',
                    f'{flip_rate*100:.4f}', run + 1, best_i + 1] + acc_values +
                   [f'{best_acc*100:.4f}', int(found[best_i])])
            writer.writerow(row)

        if all(found):
            print(f"\nAll {k_up} upper APUFs found after {run+1} runs.")
            break
    else:
        print(f"\nReached max runs ({max_runs}). Found: {sum(found)}/{k_up}")

    # ── Write summary to file ──────────────────────────────────────────────────
    summary_file = f'pclpuf_reliability_attack_KUP{k_up}_Kd{k_d}_SUMMARY.txt'
    with open(summary_file, 'a') as f:
        f.write("\n" + "="*60 + "\n")
        f.write(f"PC-LPUF Reliability Attack Summary\n")
        f.write(f"Time: {np.datetime64('now')}\n")
        f.write("="*60 + "\n")
        f.write(f"Configuration:\n")
        f.write(f"  chal_size={chal_size}, k_up={k_up}, k_d={k_d}\n")
        f.write(f"  sigma_noise_apuf={sigma_noise_apuf}, sigma_noise_reap={sigma_noise_reap}\n")
        f.write(f"  n_train={n_train}, n_test={n_test}, evaluations={evaluations}\n")
        f.write(f"  threshold={threshold}, max_runs={max_runs}\n")
        f.write(f"\nResults:\n")
        f.write(f"  PC-LPUF Reliability: {reliability*100:.2f}%\n")
        f.write(f"  Correlation (upper): {corr_up:.6f}\n")
        f.write(f"  Correlation (lower): {corr_reap:.6f}\n")
        f.write(f"  Flip Rate (upper->output): {flip_rate*100:.2f}%\n")
        f.write(f"  Attack Success: {'YES' if all(found) else 'NO'} ({sum(found)}/{k_up} APUFs found)\n")
        f.write(f"\nPer-APUF Results:\n")
        for i in range(k_up):
            status = f"found at run {found_at_run[i]}" if found[i] else "NOT found"
            f.write(f"  APUF {i+1}: {status}\n")
        f.write(f"\nDetailed results stored in: {results_file}\n")
        f.write("="*60 + "\n")

    print("\n" + "=" * 55)
    print("Results Summary:")

    print("=" * 55)
    for i in range(k_up):
        status = f"found at run {found_at_run[i]}" if found[i] else "NOT found"
        print(f"  APUF {i+1}: {status}")
    print(f"\nConclusion: Reliability attack {'SUCCEEDS' if all(found) else 'FAILS'} "
          f"on upper layer ({sum(found)}/{k_up} APUFs modeled)")
    print(f"Flip rate (upper->output): {flip_rate*100:.2f}%")
    print(f"\nDetailed results saved to: {results_file}")
    print(f"Summary saved to: {summary_file}")
    print("=" * 55 + "\n")


In [ ]:
##########XOR PUF Reliability Attack Script (CMA-ES)##########

# This script implements a reliability-based CMA-ES attack against an XOR PUF.
# It generates the underlying APUF components, simulates noisy evaluations,
# and uses reliability information to recover the individual APUF models.

import numpy as np
from cma import CMAEvolutionStrategy

# ─── XOR PUF Generation ───────────────────────────────────────────────────────
def xor_puf_generation(n_xor, chal_size, mu=0, sigma=1):
    return np.random.normal(mu, sigma, (n_xor, chal_size + 1))

# ─── Parity Transform ─────────────────────────────────────────────────────────
def transform(challenges, n_rows, chal_size):
    c = 1 - 2 * challenges.astype(float)
    Phi = np.ones((n_rows, chal_size + 1))
    for i in range(chal_size):
        Phi[:, i] = np.prod(c[:, i:], axis=1)
    return Phi

# ─── Compute XOR PUF Response ─────────────────────────────────────────────────
def compute_response_xor(xor_w, n_xor, Phi, n_rows, size):
    sums = Phi @ xor_w.T
    prod = np.prod(sums, axis=1)
    return np.where(prod > 0, 0, 1)

# ─── Compute Noisy Responses ──────────────────────────────────────────────────
def compute_noisy_responses_xor(xor_w, n_xor, Phi, n_rows, size,
                                 chal_size, sigma, sigma_noise, evaluations):
    responses = np.ones((n_rows, evaluations), dtype=int)
    for e in range(evaluations):
        noise   = np.random.normal(0, sigma_noise * sigma, (n_xor, chal_size + 1))
        noisy_w = xor_w + noise
        sums    = Phi @ noisy_w.T
        prod    = np.prod(sums, axis=1)
        responses[:, e] = np.where(prod > 0, 0, 1)
    return responses

# ─── Compute Noise Information ────────────────────────────────────────────────
def compute_noise_information(responses, n_rows, evaluations):
    row_sums = responses.sum(axis=1).astype(float)
    return np.abs(evaluations / 2.0 - row_sums)

# ─── Model Accuracy vs noiseless ground truth ────────────────────────────────
def model_acc(w, Phi, responses, n_rows):
    preds = np.where(Phi @ w > 0, 0, 1)
    return np.mean(preds == responses)

# ─── Fitness Function ─────────────────────────────────────────────────────────
def model_acc_rel(w, Phi, inform):
    delta = Phi @ w
    corr  = np.corrcoef(np.abs(delta), inform)[0, 1]
    return 0.0 if np.isnan(corr) else corr

# ─── CMA-ES ───────────────────────────────────────────────────────────────────
def cmaes_attack(Phi, inform, size, stop_eval=30000, seed=None):
    if seed is not None:
        np.random.seed(seed)
    x0 = np.random.rand(size)
    es = CMAEvolutionStrategy(x0, 0.5, {
        'maxfevals': stop_eval,
        'tolx':      1e-10,
        'verbose':   -9,
    })
    while not es.stop():
        solutions = es.ask()
        fitnesses = [-model_acc_rel(np.array(s), Phi, inform) for s in solutions]
        es.tell(solutions, fitnesses)
    return np.array(es.result.xbest)


# ─── Reliability Evaluation ──────────────────────────────────────────────────

def compute_apuf_reliability(xor_w, apuf_index, Phi, n_rows, chal_size,
                              sigma, sigma_noise, eval_reps=1000):
    """
    Reliability of a single APUF component:
    Rel = fraction of challenges where noisy response == noiseless response,
    averaged over eval_reps noise draws.
    """
    w_single = xor_w[apuf_index:apuf_index+1]
    ref = np.where((Phi @ w_single.T).flatten() > 0, 0, 1)

    matches = np.zeros(n_rows, dtype=float)
    for _ in range(eval_reps):
        noise   = np.random.normal(0, sigma_noise * sigma, (1, chal_size + 1))
        noisy_w = w_single + noise
        noisy_r = np.where((Phi @ noisy_w.T).flatten() > 0, 0, 1)
        matches += (noisy_r == ref).astype(float)

    return matches.mean() / eval_reps


def compute_xor_reliability(xor_w, n_xor, Phi, n_rows, chal_size,
                             sigma, sigma_noise, eval_reps=1000):
    """
    Reliability of the full XOR PUF:
    Rel = fraction of challenges where noisy XOR response == noiseless XOR response,
    averaged over eval_reps noise draws.
    """
    ref_xor = compute_response_xor(xor_w, n_xor, Phi, n_rows, chal_size + 1)

    matches = np.zeros(n_rows, dtype=float)
    for _ in range(eval_reps):
        noise   = np.random.normal(0, sigma_noise * sigma, (n_xor, chal_size + 1))
        noisy_w = xor_w + noise
        sums    = Phi @ noisy_w.T
        prod    = np.prod(sums, axis=1)
        noisy_r = np.where(prod > 0, 0, 1)
        matches += (noisy_r == ref_xor).astype(float)

    return matches.mean() / eval_reps


def majority_vote_response(xor_w, n_xor, Phi, n_rows, chal_size,
                            sigma, sigma_noise, eval_reps):
    """
    Majority-vote response over eval_reps noisy evaluations — the fair
    accuracy target a real attacker would estimate.
    """
    votes = np.zeros(n_rows, dtype=int)
    for _ in range(eval_reps):
        noise   = np.random.normal(0, sigma_noise * sigma, (n_xor, chal_size + 1))
        noisy_w = xor_w + noise
        sums    = Phi @ noisy_w.T
        prod    = np.prod(sums, axis=1)
        votes  += np.where(prod > 0, 0, 1)
    return (votes > eval_reps / 2).astype(int)


# ─── Main ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Experiment configuration parameters
    # chal_size: number of challenge bits per APUF component
    # mu, sigma: mean and standard deviation used to generate the APUF weights
    # n_xor: number of APUF components combined in the XOR PUF
    # n_train: number of training challenges used for the attack
    # n_test: number of test challenges used for evaluation
    # evaluations: number of noisy evaluations used to build the reliability signal
    # sigma_noise: strength of the additive Gaussian noise
    # max_runs: maximum number of CMA-ES attack runs to execute
    # threshold: accuracy threshold used to declare a component recovered
    # size: dimension of each APUF weight vector
    # rel_reps: number of noisy repetitions used for reliability estimation
    # in the paper we evaluated the following PUF: 
    # chal_size = 128, n_xor = 3
    
    chal_size    = 128
    mu           = 0
    sigma        = 1
    n_xor        = 3
    n_train      = 40000
    n_test       = 500
    evaluations  = 11
    sigma_noise  = 0.1
    max_runs     = 30
    threshold    = 0.9
    size         = chal_size + 1
    rel_reps     = 1000

    np.random.seed(10)

    xor_w = xor_puf_generation(n_xor, chal_size, mu, sigma)

    # ── Reliability Evaluation ────────────────────────────────────────────────
    rel_size = 500
    C_rel    = np.random.randint(0, 2, (rel_size, chal_size))
    Phi_rel  = transform(C_rel, rel_size, chal_size)

    apuf_reliabilities = []
    for i in range(n_xor):
        rel_i = compute_apuf_reliability(
            xor_w, i, Phi_rel, rel_size, chal_size, sigma, sigma_noise, rel_reps
        )
        apuf_reliabilities.append(rel_i)
        print(f"  APUF {i+1} reliability : {rel_i*100:.2f}%")

    xor_reliability = compute_xor_reliability(
        xor_w, n_xor, Phi_rel, rel_size, chal_size, sigma, sigma_noise, rel_reps
    )
    print(f"  XOR PUF reliability : {xor_reliability*100:.2f}%")

    # ── Test Set ───────────────────────────────────────────────────────────────
    TeS     = np.random.randint(0, 2, (n_test, chal_size))
    Phi_TeS = transform(TeS, n_test, chal_size)

    apuf_responses = np.zeros((n_xor, n_test), dtype=int)
    for i in range(n_xor):
        apuf_responses[i] = compute_response_xor(
            xor_w[i:i+1], 1, Phi_TeS, n_test, size
        )

    xor_true = compute_response_xor(xor_w, n_xor, Phi_TeS, n_test, size)
    xor_mv   = majority_vote_response(
        xor_w, n_xor, Phi_TeS, n_test, chal_size, sigma, sigma_noise, evaluations
    )

    # ── Training Data ─────────────────────────────────────────────────────────
    TrS     = np.random.randint(0, 2, (n_train, chal_size))
    Phi_TrS = transform(TrS, n_train, chal_size)
    noisy_resp = compute_noisy_responses_xor(
        xor_w, n_xor, Phi_TrS, n_train, size,
        chal_size, sigma, sigma_noise, evaluations
    )
    inform = compute_noise_information(noisy_resp, n_train, evaluations)

    # ── CMA-ES Attack ─────────────────────────────────────────────────────────
    found        = [False] * n_xor
    found_at_run = [None]  * n_xor
    best_models  = [None]  * n_xor

    for run in range(max_runs):
        w_model = cmaes_attack(Phi_TrS, inform, size, seed=run)

        accs = []
        for i in range(n_xor):
            acc = model_acc(w_model, Phi_TeS, apuf_responses[i], n_test)
            acc = max(acc, 1 - acc)
            accs.append(acc)

        best_i   = int(np.argmax(accs))
        best_acc = accs[best_i]

        print(f"Run {run+1:3d} | Accs: {[f'{a:.3f}' for a in accs]} "
              f"| Best: APUF {best_i+1} ({best_acc:.3f})")

        if best_acc >= threshold and not found[best_i]:
            found[best_i]        = True
            found_at_run[best_i] = run + 1
            best_models[best_i]  = w_model

        if all(found):
            break

    # ── Overall Accuracy ──────────────────────────────────────────────────────
    if all(m is not None for m in best_models):
        xor_pred = np.zeros(n_test, dtype=int)
        for i in range(n_xor):
            delta = Phi_TeS @ best_models[i]
            resp  = np.where(delta > 0, 0, 1)
            if model_acc(best_models[i], Phi_TeS, apuf_responses[i], n_test) < 0.5:
                resp = 1 - resp
            xor_pred = xor_pred ^ resp

        acc_vs_noiseless = np.mean(xor_pred == xor_true)
        acc_vs_mv        = np.mean(xor_pred == xor_mv)

        print(f"\n  Overall XOR PUF accuracy vs noiseless   : {acc_vs_noiseless*100:.2f}%")
        print(f"  Overall XOR PUF accuracy vs majority-vote: {acc_vs_mv*100:.2f}%")
    else:
        print("\n  Not all APUFs found — cannot compute overall XOR PUF accuracy.")
